# v40 — PRM scores for the laptop's models (and, optionally, one 7–8B model) on Colab

v40's verdict (v38's ADAPT-RS vs ADAPT-R on other model families) needs Qwen2.5-Math-PRM-7B scores for every sample, repair and resume. The laptop cannot hold the PRM; this notebook scores the files on one T4 and saves the scores next to them on Drive, then prints the pre-registered reading.

**Before running:** upload the laptop's `pretest_v40_llama3.2-3b.json`, `pretest_v40_phi4-mini.json`, `pretest_v40_gemma3-4b.json` to **`MyDrive/MAS_SHT_v40/`**. Runtime → **T4 GPU**; Secrets: `HF_API_KEY`; **Run all**. Scoring resumes where it stopped. Download the `pretest_v40_prm_*.json` files at the end and put them next to the result files in the repo.

`MODE = 'hf'` instead draws samples for ONE 7–8B model (`HF_MODEL`) into the same folder (slow on one T4: about 3–4 min per row); run `MODE = 'prm'` afterwards for its scores too.

In [ ]:
# === Cell 1 — Colab setup ===========================================================
import base64, os, subprocess, sys

from google.colab import drive
drive.mount('/content/drive')
OUT_DIR = '/content/drive/MyDrive/MAS_SHT_v40'
os.makedirs(OUT_DIR, exist_ok=True)

# The stack of the Kaggle runs. Colab's own torch is kept.
DEPS = ['transformers>=4.44.0,<4.50', 'accelerate', 'bitsandbytes', 'openai>=1.40.0',
        'python-dotenv', 'tabulate']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + DEPS)
print('Deps installed.')

def _secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

_hf = _secret('HF_API_KEY')
if _hf:
    for _k in ('HF_API_KEY', 'HUGGINGFACE_TOKEN', 'HF_TOKEN'):
        os.environ[_k] = _hf
    print('HF token loaded.')

import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU: Runtime -> Change runtime type -> T4 GPU, then run again.')
_p = torch.cuda.get_device_properties(0)
print(f'cuda:0 {_p.name}  sm_{_p.major}{_p.minor}  {_p.total_memory / 2**30:.0f} GB')
if (_p.major, _p.minor) < (7, 5):
    raise RuntimeError(f'{_p.name} is older than sm_75; 4-bit NF4 needs a T4 or newer.')

# The token never enters a URL (git would store it in .git/config); a one-off
# http.extraHeader is not persisted. The repository is public: no token is needed.
REPO_DIR = '/content/llm_thesis'
BRANCH   = 'main'
REPO_URL = 'https://github.com/marios4371/llm_thesis.git'
_tok = _secret('GITHUB_TOKEN')
_auth, _hide = [], []
if _tok:
    _hdr = base64.b64encode(f'x-access-token:{_tok}'.encode()).decode()
    _auth, _hide = ['-c', f'http.extraHeader=Authorization: Basic {_hdr}'], [_tok, _hdr]

def _git(*args):
    _r = subprocess.run(['git', *_auth, *args], capture_output=True, text=True)
    if _r.returncode:
        _err = _r.stderr
        for _s in _hide:
            _err = _err.replace(_s, '***')
        raise RuntimeError(f"git {' '.join(args)} failed:\n" + _err)
    return _r.stdout.strip()

if not os.path.isdir(REPO_DIR):
    _git('clone', REPO_URL, REPO_DIR)
_git('-C', REPO_DIR, 'remote', 'set-url', 'origin', REPO_URL)
_git('-C', REPO_DIR, 'fetch', 'origin', BRANCH)
_git('-C', REPO_DIR, 'checkout', '-B', BRANCH, f'origin/{BRANCH}')
os.chdir(REPO_DIR)
print('code at', _git('log', '-1', '--format=%h %s'))
OUT = OUT_DIR
print('files live in', OUT_DIR)
print('results go to', OUT)

In [ ]:
# === Cell 2 — offline tests (they stop the run if a frozen file changed) ==================
import subprocess, sys
for _f in ('test_v40.py',):
    _t = subprocess.run([sys.executable, _f], capture_output=True, text=True)
    print(_f, '->', _t.stdout.strip().splitlines()[-1] if _t.stdout.strip() else _t.stderr[-2000:])
    if _t.returncode != 0:
        print(_t.stdout[-5000:], _t.stderr[-3000:])
        raise RuntimeError(f'{_f} failed: do NOT start the GPU run')

In [ ]:
# === Cell 3 — PRM scores (MODE = 'prm') or one 7-8B model (MODE = 'hf') ========================
MODE = 'prm'
HF_MODEL = 'deepseek-ai/deepseek-math-7b-rl'
if MODE == 'prm':
    CMD = [sys.executable, '-u', 'pretest_v40.py', '--score-prm', '--inputs', f'{OUT_DIR}/pretest_v40_*.json']
else:
    CMD = [sys.executable, '-u', 'pretest_v40.py', '--backend', 'hf', '--models', HF_MODEL,
           '--dir', OUT_DIR, '--max-hours', '11.0']
import subprocess, sys
_p = subprocess.Popen(CMD, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for _line in _p.stdout:
    print(_line, end='')
_p.wait()
print('exit code', _p.returncode)

In [ ]:
# === Cell 4 — the pre-registered reading (no GPU) ======================================
!python -u pretest_v40.py --summary-only --dir "$OUT_DIR"